In [10]:
import sqlite3
import pandas as pd
import datetime as dt

# Connect to the database
conn = sqlite3.connect('Chinook_Sqlite.sqlite')

print("Database connected successfully!")

Database connected successfully!


In [7]:
# Set a "snapshot date" to simulate the analysis happening the day after the last invoice
snapshot_date_query = "SELECT MAX(InvoiceDate) FROM Invoice"
snapshot_date = pd.read_sql_query(snapshot_date_query, conn).iloc[0,0]

rfm_query = f"""
WITH Customer_Stats AS (
    SELECT
        c.CustomerId,
        c.FirstName || ' ' || c.LastName as CustomerName,
        c.Country,
        COUNT(i.InvoiceId) as Frequency,
        SUM(i.Total) as Monetary,
        MAX(i.InvoiceDate) as LastPurchaseDate
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY 1, 2, 3
)

SELECT
    CustomerId,
    CustomerName,
    Country,
    Frequency,
    Monetary,
    LastPurchaseDate,
    CAST(julianday('{snapshot_date}') - julianday(LastPurchaseDate) AS INT) as Recency_Days
FROM Customer_Stats
ORDER BY Monetary DESC;
"""

df_rfm = pd.read_sql_query(rfm_query, conn)

# Assigning "Scores" (1-4)
df_rfm['R_Score'] = pd.qcut(df_rfm['Recency_Days'], 4, labels=[4, 3, 2, 1]) # Recent = Higher Score
df_rfm['F_Score'] = pd.qcut(df_rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]) # Frequent = Higher Score

# Added .rank(method='first') to handle duplicate dollar amounts
df_rfm['M_Score'] = pd.qcut(df_rfm['Monetary'].rank(method='first'), 4, labels=[1, 2, 3, 4]) 

# Create the "Segment" Label
def segment_customer(row):
    # Ensure scores are integers for comparison
    r = int(row['R_Score'])
    f = int(row['F_Score'])
    
    if r >= 3 and f >= 3:
        return 'VIP / Loyal'
    elif r >= 3 and f < 3:
        return 'New Potential'
    elif r < 3 and f >= 3:
        return 'At Risk' # Used to buy often, has not lately
    else:
        return 'Lost Customer'

df_rfm['Segment'] = df_rfm.apply(segment_customer, axis=1)

# Display the distinct segments
print(df_rfm[['CustomerName', 'Country', 'Segment', 'Monetary']].head(10))

         CustomerName         Country        Segment  Monetary
0         Helena Holý  Czech Republic  New Potential     49.62
1  Richard Cunningham             USA  Lost Customer     47.62
2          Luis Rojas           Chile  Lost Customer     46.62
3     Ladislav Kovács         Hungary  New Potential     45.62
4       Hugh O'Reilly         Ireland  New Potential     45.62
5       Frank Ralston             USA  New Potential     43.62
6       Julia Barnett             USA  Lost Customer     43.62
7     Fynn Zimmermann         Germany  Lost Customer     43.62
8       Astrid Gruber         Austria  Lost Customer     42.62
9      Victor Stevens             USA  New Potential     42.62


In [9]:
df_rfm.to_csv('Customer_RFM_Segments.csv', index=False)
print("CSV saved!")

CSV saved!
